# Deep Learning Science: MobileNetV2 Architecture

## 1. Introduction
Deep Learning models, particularly Convolutional Neural Networks (CNNs), are the gold standard for image classification. For our skin cancer detection task, we use **MobileNetV2**. This notebook explains the mathematical and architectural reasons for this choice.

## 2. Why MobileNetV2? The Efficiency-Accuracy Tradeoff
Medical AI often needs to run on local devices (edge computing) for privacy and speed. Standard models like ResNet50 are heavy.

MobileNetV2 introduces **Depthwise Separable Convolutions**, which reduce the computational cost (FLOPs) by nearly 8-9x compared to standard convolutions with minimal loss in accuracy.

## 3. Mathematical Foundation

### 3.1. Standard Convolution
A standard convolution layer takes an input of size $h \times w \times d_{in}$ and produces an output $h \times w \times d_{out}$ using a kernel $k \times k$.
**Cost:** $h \cdot w \cdot d_{in} \cdot d_{out} \cdot k \cdot k$

### 3.2. Depthwise Separable Convolution
MobileNet splits this into two steps:
1. **Depthwise Convolution:** Applies a single filter per input channel.  
   **Cost:** $h \cdot w \cdot d_{in} \cdot k \cdot k$
2. **Pointwise Convolution:** A $1 \times 1$ convolution to combine the features.  
   **Cost:** $h \cdot w \cdot d_{in} \cdot d_{out} \cdot 1 \cdot 1$

**Total Cost:** $h \cdot w \cdot d_{in} \cdot (k^2 + d_{out})$

ratio $\approx \frac{1}{d_{out}} + \frac{1}{k^2}$. For a $3 \times 3$ kernel, this is roughly **1/9th** the computation.

## 4. Architectural Innovation: The Inverted Residual Block

ResNet uses "wide $\to$ narrow $\to$ wide" blocks. MobileNetV2 flips this to **"narrow $\to$ wide $\to$ narrow"**:
1. **Expansion:** $1\times1$ conv expands low-dimensional input to high-dimension.
2. **Depthwise Conv:** Lightweight filtering in high-dimension space.
3. **Projection:** $1\times1$ conv projects back to low-dimension.

This preserves information flow through "Linear Bottlenecks" (removing ReLU at the end of the block to prevent information loss).

## 5. Visualizing Our Implementation
Let's look at the model we built in `src.skin_cancer_detection.model`.

In [ ]:
import sys
import os
import tensorflow as tf

# Import package
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from skin_cancer_detection import model, config

# Build the model
cnn_model = model.create_model()

# Print Summary
cnn_model.summary()

## 6. Transfer Learning Strategy

We initialized this model with `weights='imagenet'`. 
- **Feature Extraction:** The base layers have learned to detect edges, textures, and curves from millions of generic images (ImageNet).
- **Fine-Tuning:** The top layers (`GlobalAveragePooling`, `Dense`) are random and trained by us to map those generic features specifically to skin lesions (Melanoma vs Nevis).

By freezing the base (`base_model.trainable = False`), we save time and data requirements.